In [46]:
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import os
from tqdm import tqdm
    # 设置最多显示200行和50列（无省略号）
pd.set_option('display.max_rows', 200)      # 最大行数
pd.set_option('display.max_columns', 50)    # 最大列数
pd.set_option('display.width', None)        # 自动适应终端宽度
pd.set_option('display.max_colwidth', None) # 列内容不被截断（可选）

import numpy as np

In [47]:
import pandas as pd, glob, datetime as dt
import os
dictionary = '../../crypto_futures_data/SOLUSDT/'
start, end = dt.date(2024,8,1), dt.date(2025,8,24)
df = pd.concat([pd.read_csv(f).assign(date=f[-14:-4])
                for f in glob.glob(os.path.join(dictionary, 'SOLUSDT-bookDepth-*.csv'))
                if start <= dt.datetime.strptime(f[-14:-4], '%Y-%m-%d').date() <= end])

In [ ]:
# root_dir = '../../crypto_futures_data/SOLUSDT/'
# def loadingfeaturetbl(sampleDate):
#     sample_1min_df = pd.read_csv(sampleDate)
#     return sample_1min_df
# csv_files = glob.glob(f"{root_dir}SOLUSDT-1m-*.csv")
# Process_1min_df = Parallel(n_jobs=min(10, len(csv_files)), verbose=10)(
#     delayed(loadingfeaturetbl)(file_path) for file_path in tqdm(csv_files, desc="加载文件")
# )   
# feature_1min_k = pd.concat(Process_1min_df, ignore_index=True)

# feature_1min_k['close_time'] = pd.to_datetime(feature_1min_k['close_time'], unit='ms')
# df['close_time'] = pd.to_datetime(df['timestamp']).dt.floor('1min') 
# feature_1min_k['close_time'] = pd.to_datetime(feature_1min_k['close_time']).dt.floor('1min') 
# df = df.merge(feature_1min_k, left_on='close_time', right_on='close_time', how='left')

In [51]:
df = pd.read_csv('../../crypto_futures_data/SOLUSDT/SOLUSDT-bookDepth-2025-08-11.csv')

In [52]:
df['mid_price'] = (df['avg_price_1'] + df['avg_price_-1']) / 2
df.timestamp = pd.to_datetime(df.timestamp)


- 挂单的筹码分布skew 时间做一个decay，偏度的变化量，（全量订单簿变化量的偏度， 最后一个时刻的偏度）
- 挂单的筹码：价格上涨：新增档位委买量+原档位净委买增加量+主买成交额； 被动卖出成交额+原档位净委卖减少量
## 
- 新增档位委买量 / 被动卖出成交额；
- 1.（新增档位委买量 + 净主买成交额） /  净被动卖出成交额
- 2.（新增档位委买量 + 原档位净委买增加量） / （被动卖出成交额 + 原档位净委卖减少量）
- 3.（新增档位委买量 + 原档位净委买增加量 + 净主买成交额） / （净被动卖出成交额 + 原档位净委卖减少量）

全量订单簿变化量



In [53]:
df['ts_1min'] = df.timestamp.dt.ceil('1min')
Omin_df = df.groupby(['ts_1min'])['mid_price'].last()

OminLag60s_df = Omin_df.shift(-1)
OminLag180s_df = Omin_df.shift(-3)
OminLag300s_df = Omin_df.shift(-5)
OminLag600s_df = Omin_df.shift(-10)
OminLag1800s_df = Omin_df.shift(-30)
OminLag3600s_df = Omin_df.shift(-60)
OminLag_df = pd.concat([OminLag60s_df, OminLag180s_df, OminLag300s_df, OminLag600s_df, OminLag1800s_df, OminLag3600s_df], axis=1)

OminLag_df.columns=[f'mid_price_lag{lag}s' for lag in [60, 180, 300, 600, 1800, 3600]]

df_merge = df.merge(OminLag_df, on=['ts_1min'], how='left')

for lag in [60, 180, 300, 600, 1800, 3600]:
    df_merge[f'LA{lag}'] = (df_merge[f'mid_price_lag{lag}s'] / df_merge['mid_price'] - 1.) * 10000
    
# df_merge['Imb'] = (df_merge['avg_price_-1'] - df_merge['avg_price_1']) / (df_merge['avg_price_-1'] + df_merge['avg_price_1'])
# df_merge['bidsum4'] = df_merge[['avg_price_-1','avg_price_-2', 'avg_price_-3', 'avg_price_-4']].sum(axis=1)
# df_merge['asksum4'] = df_merge[['avg_price_1','avg_price_2', 'avg_price_3', 'avg_price_4']].sum(axis=1)
# df_merge['Imb4'] = df_merge.eval('(bidsum4 - asksum4) / (bidsum4 + asksum4)')

In [ ]:

def calculate_orderbook_changes(df):
    """
    计算每个timestamp与过去100个timestamp相比的订单簿变化量
    
    参数:
    df (pd.DataFrame): 包含订单簿数据的DataFrame
    
    返回:
    pd.DataFrame: 包含计算结果的DataFrame
    """
    # 确保数据按时间排序
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    # 提取档位数量
    bid_levels = 5
    ask_levels = 5
    
    # 初始化结果DataFrame
    results = []
    
    # 遍历每个时间点
    for i in tqdm(range(100, len(df))):
        current_idx = i
        past_idx = i - 100
        
        # 获取当前和过去的订单簿数据
        current_row = df.iloc[current_idx]
        past_row = df.iloc[past_idx]
        
        # 提取当前订单簿数据
        current_bids = []
        current_asks = []
        
        for level in range(1, bid_levels + 1):
            price = round(current_row[f'avg_price_-{level}'], 2)
            depth = current_row[f'discrete_depth_-{level}']
            current_bids.append((price, depth))
            
        for level in range(1, ask_levels + 1):
            price = round(current_row[f'avg_price_{level}'], 2)
            depth = current_row[f'discrete_depth_{level}']
            current_asks.append((price, depth))
            
        # 提取过去订单簿数据
        past_bids = []
        past_asks = []
        
        for level in range(1, bid_levels + 1):
            price = round(past_row[f'avg_price_-{level}'], 2)
            depth = past_row[f'discrete_depth_-{level}']
            past_bids.append((price, depth))
            
        for level in range(1, ask_levels + 1):
            price = round(past_row[f'avg_price_{level}'], 2)
            depth = past_row[f'discrete_depth_{level}']
            past_asks.append((price, depth))
            
        # 计算买方档位变化
        bid_changes = calculate_level_changes(
            current_bids, past_bids, 
            sort_key=lambda x: -x  # 买方按价格降序
        )
        
        # 计算卖方档位变化
        ask_changes = calculate_level_changes(
            current_asks, past_asks, 
            sort_key=lambda x: x  # 卖方按价格升序
        )
        
        # 构建结果行
        result_row = {
            'timestamp': current_row['timestamp'],
            'past_timestamp': past_row['timestamp']
        }
        
        # 添加买方变化
        for level, (price, depth_diff) in enumerate(bid_changes, 1):
            result_row[f'bid_price_change_{level}'] = price
            result_row[f'bid_depth_change_{level}'] = depth_diff
            
        # 添加卖方变化
        for level, (price, depth_diff) in enumerate(ask_changes, 1):
            result_row[f'ask_price_change_{level}'] = price
            result_row[f'ask_depth_change_{level}'] = depth_diff
            
        results.append(result_row)
        
    return pd.DataFrame(results)

def calculate_level_changes(current_levels, past_levels, sort_key):
    """
    计算两个时间点之间的档位变化
    参数:
    current_levels (list): 当前档位列表，每个元素为(price, depth)元组
    past_levels (list): 过去档位列表，每个元素为(price, depth)元组
    sort_key (function): 排序函数，用于确定档位顺序
    返回:
    list: 每个档位的变化，每个元素为(price_diff, depth_diff)元组
    """
    # 创建价格到深度的映射
    current_depth_map = {price: depth for price, depth in current_levels}
    past_depth_map = {price: depth for price, depth in past_levels}

    # 获取所有唯一价格并排序
    all_prices = sorted(
        set(current_depth_map.keys()).union(set(past_depth_map.keys())),
        key=lambda x: -x  # 买方按价格降序
    )

    # 计算每个档位的变化
    changes = []
    for price in all_prices:
        current_depth = current_depth_map.get(price, 0)
        past_depth = past_depth_map.get(price, 0)
        depth_diff = current_depth - past_depth
        changes.append((price, depth_diff))
        
    return changes


In [135]:
changes_df = calculate_orderbook_changes(df)

100%|██████████| 2712/2712 [00:00<00:00, 5301.61it/s]


In [ ]:
changes_every_df = calculate_orderbook_changes_every(df)

100%|██████████| 2811/2811 [00:00<00:00, 5458.64it/s]


In [ ]:
changes_every_df

,timestamp,past_timestamp,bid_price_change_1,bid_depth_change_1,bid_price_change_2,bid_depth_change_2,bid_price_change_3,bid_depth_change_3,bid_price_change_4,bid_depth_change_4,bid_price_change_5,bid_depth_change_5,bid_price_change_6,bid_depth_change_6,bid_price_change_7,bid_depth_change_7,bid_price_change_8,bid_depth_change_8,bid_price_change_9,bid_depth_change_9,bid_price_change_10,bid_depth_change_10,ask_price_change_1,ask_depth_change_1,ask_price_change_2,ask_depth_change_2,ask_price_change_3,ask_depth_change_3,ask_price_change_4,ask_depth_change_4,ask_price_change_5,ask_depth_change_5,ask_price_change_6,ask_depth_change_6,ask_price_change_7,ask_depth_change_7,ask_price_change_8,ask_depth_change_8,ask_price_change_9,ask_depth_change_9,ask_price_change_10,ask_depth_change_10
0,2025-08-11 00:00:53,2025-08-11 00:00:07,181.81,-146894.26,181.54,162465.08,179.65,-139415.37,179.44,135228.14,178.06,-147801.66,177.87,154082.37,176.20,-76146.45,175.92,65006.29,174.25,-79774.90,174.16,76351.76,190.13,-45390.75,190.05,44756.85,188.76,-20117.77,188.18,51764.35,187.14,-133783.23,186.73,117430.33,185.58,-102223.80,185.34,93081.88,183.32,-177347.13,183.17,186391.93
1,2025-08-11 00:01:08,2025-08-11 00:00:53,181.54,-162465.08,181.46,180333.88,179.44,-135228.14,179.40,129822.16,177.88,160411.97,177.87,-154082.37,175.92,-65006.29,175.90,63076.68,174.16,-76351.76,174.15,76023.85,190.05,-5.34,188.18,-423.64,186.73,-117430.33,186.68,109385.98,185.34,8240.91,183.17,-186391.93,183.12,178116.74,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-08-11 00:01:32,2025-08-11 00:01:08,181.46,-180333.88,181.44,182525.25,179.40,2075.83,177.88,-160411.97,177.87,161404.72,175.91,63031.61,175.90,-63076.68,174.15,-76023.85,174.14,80509.13,NaN,NaN,190.05,91.19,188.18,-51340.71,188.17,51406.08,186.68,255.08,185.34,-101322.79,185.31,104538.03,183.12,-178116.74,183.11,187018.54,NaN,NaN,NaN,NaN
3,2025-08-11 00:02:01,2025-08-11 00:01:32,181.53,172751.54,181.44,-182525.25,179.48,136505.25,179.40,-131897.99,177.90,163304.93,177.87,-161404.72,175.95,64564.55,175.91,-63031.61,174.17,80128.32,174.14,-80509.13,190.08,44696.81,190.05,-44842.70,188.21,51373.96,188.17,-51406.08,186.80,95311.66,186.68,-109641.06,185.47,113206.28,185.31,-104538.03,183.19,195743.54,183.11,-187018.54
4,2025-08-11 00:02:32,2025-08-11 00:02:01,181.53,2613.83,179.48,652.80,177.90,-359.34,175.95,-64564.55,175.94,64719.75,174.17,-86.31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,190.08,-44696.81,190.06,44307.39,188.21,-51373.96,188.19,50648.13,186.80,-95311.66,186.79,96386.56,185.47,-113206.28,185.46,112630.98,183.19,-195743.54,183.17,198648.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2806,2025-08-11 23:57:33,2025-08-11 23:57:03,174.21,162530.79,174.16,-153905.53,172.05,162440.96,172.02,-166277.67,170.40,248234.81,170.37,-246195.50,168.47,75945.46,168.42,-73377.47,166.73,34168.63,166.45,-59668.61,182.51,14755.66,182.49,-14748.03,180.94,16257.55,180.86,-17462.58,179.19,85977.03,179.14,-87818.04,177.78,79571.93,177.75,-76421.06,175.62,216423.62,175.59,-216267.97
2807,2025-08-11 23:58:01,2025-08-11 23:57:33,174.21,-162530.79,174.17,156126.72,172.05,-162440.96,172.03,164875.85,170.40,-248234.81,170.38,247156.76,168.47,-75945.46,168.42,73182.94,166.73,-34168.63,166.45,60018.04,182.51,-14755.66,182.50,14690.78,180.94,-16257.55,180.91,16926.02,179.19,-85977.03,179.16,85744.60,177.78,-79571.93,177.76,77528.95,175.62,-216423.62,175.61,216013.23
2808,2025-08-11 23:58:33,2025-08-11 23:58:01,174.17,-156126.72,174.13,156871.10,172.03,-164875.85,172.00,169242.14,170.38,-247156.76,170.36,242830.19,168.42,-73182.94,168.41,73327.38,166.45,-60018.04,166.43,60841.95,182.50,-14690.78,182.47,14935.16,180.91,-16926.02,180.81,17943.12,179.16,-85744.60,179.09,91306.01,177.76,-77528.95,177.69,73573.36,175.61,-216013.23,175.57,214636.70
2809,2025-08-11 23:59:03,2025-08-11 23:58:33,174.13,-156871.10,174.04,159643.02,172.00,-1692

In [139]:

def calculate_orderbook_changes_every(df):
    """
    计算每个timestamp与过去100个timestamp相比的订单簿变化量
    
    参数:
    df (pd.DataFrame): 包含订单簿数据的DataFrame
    
    返回:
    pd.DataFrame: 包含计算结果的DataFrame
    """
    # 确保数据按时间排序
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    # 提取档位数量
    bid_levels = 5
    ask_levels = 5
    
    # 初始化结果DataFrame
    results = []
    

    # 遍历每个时间点
    for i in tqdm(range(1, len(df))):
        current_idx = i
        past_idx = i - 1
        
        # 获取当前和过去的订单簿数据
        current_row = df.iloc[current_idx]
        past_row = df.iloc[past_idx]
        
        # 提取当前订单簿数据
        current_bids = []
        current_asks = []
        
        for level in range(1, bid_levels + 1):
            price = round(current_row[f'avg_price_-{level}'], 2)
            depth = current_row[f'discrete_depth_-{level}']
            current_bids.append((price, depth))
            
        for level in range(1, ask_levels + 1):
            price = round(current_row[f'avg_price_{level}'], 2)
            depth = current_row[f'discrete_depth_{level}']
            current_asks.append((price, depth))
            
        # 提取过去订单簿数据
        past_bids = []
        past_asks = []
        
        for level in range(1, bid_levels + 1):
            price = round(past_row[f'avg_price_-{level}'], 2)
            depth = past_row[f'discrete_depth_-{level}']
            past_bids.append((price, depth))
            
        for level in range(1, ask_levels + 1):
            price = round(past_row[f'avg_price_{level}'], 2)
            depth = past_row[f'discrete_depth_{level}']
            past_asks.append((price, depth))
        

            
        # 计算买方档位变化
        bid_changes = calculate_level_changes(
            current_bids, past_bids, 
            sort_key=lambda x: -x  # 买方按价格降序
        )
        
        # 计算卖方档位变化
        ask_changes = calculate_level_changes(
            current_asks, past_asks, 
            sort_key=lambda x: x  # 卖方按价格升序
        )
        
        # 构建结果行
        result_row = {
            'timestamp': current_row['timestamp'],
            'past_timestamp': past_row['timestamp']
        }
        
        # 添加买方变化
        for level, (price, depth_diff) in enumerate(bid_changes, 1):
            result_row[f'bid_price_change_{level}'] = price
            result_row[f'bid_depth_change_{level}'] = depth_diff
            
        # 添加卖方变化
        for level, (price, depth_diff) in enumerate(ask_changes, 1):
            result_row[f'ask_price_change_{level}'] = price
            result_row[f'ask_depth_change_{level}'] = depth_diff
            
        results.append(result_row)
        
    return pd.DataFrame(results)

def calculate_level_changes(current_levels, past_levels, sort_key):
    """
    计算两个时间点之间的档位变化
    参数:
    current_levels (list): 当前档位列表，每个元素为(price, depth)元组
    past_levels (list): 过去档位列表，每个元素为(price, depth)元组
    sort_key (function): 排序函数，用于确定档位顺序
    返回:
    list: 每个档位的变化，每个元素为(price_diff, depth_diff)元组
    """
    # 创建价格到深度的映射
    current_depth_map = {price: depth for price, depth in current_levels}
    past_depth_map = {price: depth for price, depth in past_levels}

    # 获取所有唯一价格并排序
    all_prices = sorted(
        set(current_depth_map.keys()).union(set(past_depth_map.keys())),
        key=lambda x: -x  # 买方按价格降序
    )

    # 计算每个档位的变化
    changes = []
    for price in all_prices:
        current_depth = current_depth_map.get(price, 0)
        past_depth = past_depth_map.get(price, 0)
        depth_diff = current_depth - past_depth
        changes.append((price, depth_diff))
        
    return changes


In [131]:

# 遍历每个时间点
for i in tqdm(range(1, len(df))):
    current_idx = i
    past_idx = i - 1
    
    # 获取当前和过去的订单簿数据
    current_row = df.iloc[current_idx]
    past_row = df.iloc[past_idx]
    
    # 提取当前订单簿数据
    current_bids = []
    current_asks = []
    
    for level in range(1, bid_levels + 1):
        price = round(current_row[f'avg_price_-{level}'], 2)
        depth = current_row[f'discrete_depth_-{level}']
        current_bids.append((price, depth))
        
    for level in range(1, ask_levels + 1):
        price = round(current_row[f'avg_price_{level}'], 2)
        depth = current_row[f'discrete_depth_{level}']
        current_asks.append((price, depth))
        
    # 提取过去订单簿数据
    past_bids = []
    past_asks = []
    
    for level in range(1, bid_levels + 1):
        price = round(past_row[f'avg_price_-{level}'], 2)
        depth = past_row[f'discrete_depth_-{level}']
        past_bids.append((price, depth))
        
    for level in range(1, ask_levels + 1):
        price = round(past_row[f'avg_price_{level}'], 2)
        depth = past_row[f'discrete_depth_{level}']
        past_asks.append((price, depth))
    break


  0%|          | 0/2811 [00:00<?, ?it/s]


In [132]:
bid_changes = calculate_level_changes(
    current_bids, past_bids, 
    sort_key=lambda x: -x  # 买方按价格降序
)

In [ ]:
current_depth_map = {price: depth for price, depth in current_levels}
past_depth_map = {price: depth for price, depth in past_levels}

# 获取所有唯一价格并排序
all_prices = sorted(
    set(current_depth_map.keys()).union(set(past_depth_map.keys())),
    key=lambda x: -x  # 买方按价格降序
)

# 计算每个档位的变化
changes = []
for price in all_prices:
    current_depth = current_depth_map.get(price, 0)
    past_depth = past_depth_map.get(price, 0)
    depth_diff = current_depth - past_depth
    changes.append((price, depth_diff))

In [ ]:

from collections import defaultdict  # 需导入defaultdict
def calculate_orderbook_skew_separate(changes_every_df, window_size=100, decay_factor=0.99):
    """
    买卖方分离计算：每100个tick窗口内，时间加权的挂单/撤单量（净），并输出各自的skew和最大净挂单位置。
    
    参数：
    changes_every_df (pd.DataFrame)：输入数据，含每个tick相对上一tick的买卖档位变化（price+depth）。
    window_size (int)：滑动窗口大小，默认100个tick。
    decay_factor (float)：时间衰减因子（越靠近当前tick权重越高），默认0.99（上一tick权重为当前的99%）。
    
    返回：
    pd.DataFrame：每个窗口的结果，含买卖方各自的skew、最大净挂单量及位置。
    """
    # 1. 数据预处理：确保按时间排序
    df = changes_every_df.copy()
    # df['timestamp'] = pd.to_datetime(df['timestamp'])  # 确保时间格式正确
    # df = df.sort_values('timestamp').reset_index(drop=True)
    
    # 提取买卖档位列名（适配任意数量的档位，如bid_price_change_1~10）
    bid_price_cols = [col for col in df.columns if col.startswith('bid_price_change_')]
    bid_depth_cols = [col.replace('price', 'depth') for col in bid_price_cols]
    ask_price_cols = [col for col in df.columns if col.startswith('ask_price_change_')]
    ask_depth_cols = [col.replace('price', 'depth') for col in ask_price_cols]
    
    # 2. 初始化结果列表
    results = []
    
    # 3. 滑动窗口处理（每window_size个tick为一个窗口）
    for window_end_idx in range(window_size, len(df) + 1, window_size):
        # 截取当前窗口数据（从window_end_idx - window_size 到 window_end_idx - 1）
        window_df = df.iloc[window_end_idx - window_size : window_end_idx]
        window_start_ts = window_df['timestamp'].iloc[0]  # 窗口起始时间
        window_end_ts = window_df['timestamp'].iloc[-1]    # 窗口结束时间
        
        # -------------------------- 关键步骤1：计算窗口内每个tick的时间衰减权重 --------------------------
        # 权重逻辑：窗口内从“旧”到“新”，权重为 decay_factor^0, decay_factor^1, ..., decay_factor^(window_size-1)
        # 越新的tick权重越高（如window_size=3，权重为 [0.99^0, 0.99^1, 0.99^2] → 旧→新权重递增）
        weights = np.array([decay_factor ** t for t in range(len(window_df))])
        weights = weights / weights.sum()  # 归一化（可选，确保权重和为1，不影响相对大小）
        
        # -------------------------- 关键步骤2：买卖方独立累积“时间加权净挂单量” --------------------------
        # 2.1 买方（bid）累积：key=价格，value=加权后的净挂单量（正=挂单，负=撤单）
        # 用defaultdict替代普通字典，省去“if price not in dict”的判断
        bid_weighted_accum = defaultdict(float)
        ask_weighted_accum = defaultdict(float)

        # 仅遍历1次窗口数据，同时处理买卖方（减少window_size次循环）
        for idx, (_, row) in enumerate(window_df.iterrows()):
            tick_weight = weights[idx]
            
            # 1. 处理买方：一次性提取所有非NaN的价格和深度（列表推导式比逐个循环更高效）
            #  zip(bid_price_cols, bid_depth_cols) 是档位列对（如(price1, depth1), (price2, depth2)）
            bid_valid_pairs = [
                (row[p_col], row[d_col]) 
                for p_col, d_col in zip(bid_price_cols, bid_depth_cols) 
                if pd.notna(row[p_col])  # 过滤NaN价格
            ]
            # 批量累加：避免内层循环的重复键检查（defaultdict直接赋值）
            for price, net_depth in bid_valid_pairs:
                bid_weighted_accum[price] += net_depth * tick_weight
            
            # 2. 处理卖方：逻辑同买方，复用同一轮tick循环
            ask_valid_pairs = [
                (row[p_col], row[d_col]) 
                for p_col, d_col in zip(ask_price_cols, ask_depth_cols) 
                if pd.notna(row[p_col])
            ]
            for price, net_depth in ask_valid_pairs:
                ask_weighted_accum[price] += net_depth * tick_weight
        
        # -------------------------- 关键步骤3：买卖方独立补全价格序列（0.01步长） --------------------------
        def get_full_price_sequence(price_dict, step=0.01):
            """辅助函数：基于输入价格字典，生成0.01步长的完整价格序列及对应净挂单量"""
            if not price_dict:  # 该方无任何数据
                return np.array([]), np.array([])
            # 确定价格范围（避免浮点数精度问题，转整数处理）
            min_p = min(price_dict.keys())
            max_p = max(price_dict.keys())
            min_p_int = int(round(min_p * 100))  # 转成“分”为单位的整数
            max_p_int = int(round(max_p * 100))
            # 生成0.01步长的价格序列
            full_prices = np.array([p_int / 100 for p_int in range(min_p_int, max_p_int + 1)])
            # 匹配对应净挂单量（缺失价格补0）
            full_net_depth = np.array([price_dict.get(p, 0.0) for p in full_prices])
            return full_prices, full_net_depth
        
        # 买方补全价格序列
        bid_full_prices, bid_full_net_depth = get_full_price_sequence(bid_weighted_accum)
        # 卖方补全价格序列
        ask_full_prices, ask_full_net_depth = get_full_price_sequence(ask_weighted_accum)
        
        # -------------------------- 关键步骤4：买卖方独立计算指标（skew + 最大净挂单位置） --------------------------
        def calculate_skew_and_max(prices, net_depths):
            """辅助函数：计算单个方向（买/卖）的skew和最大净挂单量位置"""
            if len(net_depths) == 0:  # 无数据时返回NaN
                return np.nan, np.nan, np.nan
            
            total_net = np.sum(net_depths)
            positions = np.arange(len(net_depths))  # 价格序列的位置索引（0开始，从小到大）
            
            # 计算skew（偏度：衡量净挂单量在价格序列上的分布偏向）
            if total_net == 0:
                skew = 0.0  # 总净挂单为0，无偏向
            else:
                # 加权平均位置（以净挂单量为权重）
                weighted_mean_pos = np.sum(net_depths * positions) / total_net
                # 加权标准差
                weighted_var = np.sum(net_depths * (positions - weighted_mean_pos) ** 2) / total_net
                weighted_std_pos = np.sqrt(weighted_var) if weighted_var > 0 else 1e-8
                # 标准化偏度（避免量级影响）
                skew = np.sum(net_depths * (positions - weighted_mean_pos)) / (len(positions) * weighted_std_pos)
            
            # 最大净挂单量及对应位置
            max_net_idx = np.argmax(net_depths)
            max_net_value = net_depths[max_net_idx]
            max_net_price = prices[max_net_idx]
            
            return skew, max_net_value, max_net_price, max_net_idx
        
        # 计算买方指标
        bid_skew, bid_max_net, bid_max_price, bid_max_pos = calculate_skew_and_max(bid_full_prices, bid_full_net_depth)
        # 计算卖方指标
        ask_skew, ask_max_net, ask_max_price, ask_max_pos = calculate_skew_and_max(ask_full_prices, ask_full_net_depth)
        
        # -------------------------- 保存当前窗口结果 --------------------------
        results.append({
            'window_start_timestamp': window_start_ts,
            'window_end_timestamp': window_end_ts,
            # 买方结果
            'bid_skew': bid_skew,
            'bid_max_net_order': bid_max_net,
            'bid_max_net_price': bid_max_price,
            'bid_max_net_position': bid_max_pos,
            # 卖方结果
            'ask_skew': ask_skew,
            'ask_max_net_order': ask_max_net,
            'ask_max_net_price': ask_max_price,
            'ask_max_net_position': ask_max_pos,
            # 可选：保存补全后的价格和净挂单序列（便于后续分析）
            'bid_full_prices': bid_full_prices if len(bid_full_prices) > 0 else None,
            'bid_full_net_depths': bid_full_net_depth if len(bid_full_net_depth) > 0 else None,
            'ask_full_prices': ask_full_prices if len(ask_full_prices) > 0 else None,
            'ask_full_net_depths': ask_full_net_depth if len(ask_full_net_depth) > 0 else None
        })
    
    # 转换结果为DataFrame
    return pd.DataFrame(results)


# -------------------------- 示例用法 --------------------------
if __name__ == '__main__':
    # 1. 模拟用户提供的changes_every_df（使用样例数据）
    sample_data = [
        {
            'timestamp': '2025-08-11 00:00:53',
            'past_timestamp': '2025-08-11 00:00:07',
            'bid_price_change_1': 181.81, 'bid_depth_change_1': -146894.26,
            'bid_price_change_2': 181.54, 'bid_depth_change_2': 162465.08,
            'bid_price_change_3': 179.65, 'bid_depth_change_3': -139415.37,
            'ask_price_change_1': 190.13, 'ask_depth_change_1': -45390.75,
            'ask_price_change_2': 190.05, 'ask_depth_change_2': 44756.85,
            'ask_price_change_3': 188.76, 'ask_depth_change_3': -20117.77
        },
        {
            'timestamp': '2025-08-11 00:01:08',
            'past_timestamp': '2025-08-11 00:00:53',
            'bid_price_change_1': 181.54, 'bid_depth_change_1': -162465.08,
            'bid_price_change_2': 181.46, 'bid_depth_change_2': 180333.88,
            'bid_price_change_3': 179.44, 'bid_depth_change_3': -135228.14,
            'ask_price_change_1': 190.05, 'ask_depth_change_1': -5.34,
            'ask_price_change_2': 188.18, 'ask_depth_change_2': -423.64,
            'ask_price_change_3': 186.73, 'ask_depth_change_3': -117430.33
        }
    ]
    # 扩展到200条数据（满足window_size=100的测试需求，实际用用户自己的数据即可）
    extended_data = sample_data * 100
    changes_every_df = pd.DataFrame(extended_data)
    
    # 2. 调用函数计算（窗口大小设为100，衰减因子0.99）
    result_df = calculate_orderbook_skew_separate(
        changes_every_df=changes_every_df,
        window_size=100,
        decay_factor=0.99
    )
    
    # 3. 查看结果（重点看买卖方各自的skew和最大净挂单位置）
    print("窗口结果预览（前2列关键指标）：")
    print(result_df[['window_start_timestamp', 'window_end_timestamp', 'bid_skew', 'ask_skew', 'bid_max_net_position', 'ask_max_net_position']].head())

In [ ]:
df_merge['order_book_id'] = 'SOLUSDTSWAP'

In [ ]:
def plot_cols(feat_col, feature_all_df):
    # # 假设你已经有以下变量
    # feat_col = 'Around3s_AvgPrc_minusclose'
    ret_cols = ['LA300', 'LA1800']


    df = feature_all_df.reset_index()

    # ---------- 全部时间点 (FR_all_*) ----------
    fr_all_results = {}
    for ret_col in ret_cols:
        sub_df = df[[feat_col, ret_col, 'date']].dropna()

        # 按date groupby，计算 FR = cov(x, y) / std(x)
        grouped = sub_df.groupby('date')
        mean_x = grouped[feat_col].transform('mean')
        mean_y = grouped[ret_col].transform('mean')
        std_x = grouped[feat_col].transform('std')

        cov_xy = ((sub_df[feat_col] - mean_x) * (sub_df[ret_col] - mean_y)).groupby(sub_df['date']).mean()
        std_x_daily = std_x.groupby(sub_df['date']).first()  # std(x)是一样的，随便取一个就行

        fr = cov_xy / std_x_daily.replace(0, np.nan)
        fr_all_results[f'FR_all_{ret_col}'] = fr

    # ---------- 特定时间点 (FR_{time}_{ret_col}) ----------
    fr_time_results = {}


    # 合并所有结果
    fr_df = pd.DataFrame(fr_all_results)
    cumulative_fr_df = fr_df.cumsum()

    # 分别提取 ret0e 和 ret1h 的 FR 列
    fr_0e_cols = [col for col in cumulative_fr_df.columns if 'LA300' in col]
    fr_1h_cols = [col for col in cumulative_fr_df.columns if 'LA1800' in col]
    

    # 画 ret0e_shift 的 Cumulative FR 图
    plt.figure(figsize=(16, 6))
    for col in fr_0e_cols:
        plt.plot(cumulative_fr_df.index, cumulative_fr_df[col], label=col)

    plt.title(f'{feat_col} Cumulative FR (LA300_shift)')
    plt.xlabel('Date')
    plt.ylabel('Cumulative FR')
    plt.legend()
    plt.grid(True)
    plt.xticks(cumulative_fr_df.index[::10], rotation=45)
    plt.tight_layout()
    plt.show()

    # 画 ret1h_shift 的 Cumulative FR 图
    plt.figure(figsize=(16, 6))
    for col in fr_1h_cols:
        plt.plot(cumulative_fr_df.index, cumulative_fr_df[col], label=col)

    plt.title(f'{feat_col} Cumulative FR (LA1800_shift) -|')
    plt.xlabel('Date')
    plt.ylabel('Cumulative FR')
    plt.legend()
    plt.grid(True)
    plt.xticks(cumulative_fr_df.index[::10], rotation=45)
    plt.tight_layout()
    plt.show()
#     fr_df.to_parquet(f'/data/beef3/mike/pshared/GAlpha_custom_build/wensheng_jawN9J/fr_save/{feat_col}.parquet', engine="pyarrow", compression="snappy")
    


In [ ]:
for col in  ['hi']:
    plot_cols(df_results_imbreact, col)

In [ ]:
我已经构造了一个L3订单簿类，现在需要扩充功能，即原来删除订单是直接将对应的订单从队列中remove 现在是把这个订单的OrderID变更为原来的1000倍（表示订单已经被删除，只有从删除机制中查找才能找到）  但仍然保留在队列中供后续使用，原来减少订单是直接将原订单size 减少，现在减少订单是除了将原来订单size减少到指定值外，还在减少的订单位置后面增加一个消失的订单，其订单号为原订单号*1000，size为减少的量。

请你修改这个类，完善上述功能。

In [ ]:
	timestamp	past_timestamp	bid_price_change_1	bid_depth_change_1	bid_price_change_2	bid_depth_change_2	bid_price_change_3	bid_depth_change_3	bid_price_change_4	bid_depth_change_4	bid_price_change_5	bid_depth_change_5	bid_price_change_6	bid_depth_change_6	bid_price_change_7	bid_depth_change_7	bid_price_change_8	bid_depth_change_8	bid_price_change_9	bid_depth_change_9	bid_price_change_10	bid_depth_change_10	ask_price_change_1	ask_depth_change_1	ask_price_change_2	ask_depth_change_2	ask_price_change_3	ask_depth_change_3	ask_price_change_4	ask_depth_change_4	ask_price_change_5	ask_depth_change_5	ask_price_change_6	ask_depth_change_6	ask_price_change_7	ask_depth_change_7	ask_price_change_8	ask_depth_change_8	ask_price_change_9	ask_depth_change_9	ask_price_change_10	ask_depth_change_10
0	2025-08-11 00:00:53	2025-08-11 00:00:07	181.81	-146894.26	181.54	162465.08	179.65	-139415.37	179.44	135228.14	178.06	-147801.66	177.87	154082.37	176.20	-76146.45	175.92	65006.29	174.25	-79774.90	174.16	76351.76	190.13	-45390.75	190.05	44756.85	188.76	-20117.77	188.18	51764.35	187.14	-133783.23	186.73	117430.33	185.58	-102223.80	185.34	93081.88	183.32	-177347.13	183.17	186391.93
1	2025-08-11 00:01:08	2025-08-11 00:00:53	181.54	-162465.08	181.46	180333.88	179.44	-135228.14	179.40	129822.16	177.88	160411.97	177.87	-154082.37	175.92	-65006.29	175.90	63076.68	174.16	-76351.76	174.15	76023.85	190.05	-5.34	188.18	-423.64	186.73	-117430.33	186.68	109385.98	185.34	8240.91	183.17	-186391.93	183.12	178116.74	NaN	NaN	NaN	NaN	NaN	NaN
2	2025-08-11 00:01:32	2025-08-11 00:01:08	181.46	-180333.88	181.44	182525.25	179.40	2075.83	177.88	-160411.97	177.87	161404.72	175.91	63031.61	175.90	-63076.68	174.15	-76023.85	174.14	80509.13	NaN	NaN	190.05	91.19	188.18	-51340.71	188.17	51406.08	186.68	255.08	185.34	-101322.79	185.31	104538.03	183.12	-178116.74	183.11	187018.54	NaN	NaN	NaN	NaN

我现在有如上的changes_every_df，显示对任意timestamp相对上一个timestamp全量订单簿的变化量（近似可以看做当前档位上的挂单/撤单），我现在希望每100个tick， 帮我用时间加权的方式，给出买卖各个价位的挂单/撤单的decay加权和（过去的量权重低，更靠近现在的量权重高）最后计算出卖方，买方净挂单量的skew（注意，需要先将买卖方价格从小到大排序，然后以0.01（最小ticksize）为单位，产生depth_change 序列，注意补全其中不存在的价格，即要求从最小价格到最大价格每隔0.01的价格都存在对应的change_vol 如果原来不存在则补0）

同时计算最大的净挂单量在第几个价格位置处。

我现在有订单簿如下：
timestamp	avg_price_-5	avg_price_-4	avg_price_-3	avg_price_-2	avg_price_-1	avg_price_1	avg_price_2	avg_price_3	avg_price_4	avg_price_5	discrete_depth_-5	discrete_depth_-4	discrete_depth_-3	discrete_depth_-2	discrete_depth_-1	discrete_depth_1	discrete_depth_2	discrete_depth_3	discrete_depth_4	discrete_depth_5	discrete_notional_-5	discrete_notional_-4	discrete_notional_-3	discrete_notional_-2	discrete_notional_-1	discrete_notional_1	discrete_notional_2	discrete_notional_3	discrete_notional_4	discrete_notional_5	date
0	2025-06-01 00:00:09	149.333604	151.002722	152.458305	153.884236	155.636324	156.998634	158.994216	160.109773	161.863406	163.359502	90147.98	38812.33	109492.38	75229.05	139475.79	117385.02	130156.00	100361.26	31124.80	20005.08	1.346212e+07	5.860767e+06	1.669302e+07	1.157656e+07	2.170750e+07	1.842929e+07	2.069405e+07	1.606882e+07	5.037966e+06	3.268020e+06	2025-06-01
1	2025-06-01 00:01:05	149.364458	151.209421	152.592011	153.982991	155.726454	157.111082	159.071508	160.185733	161.946872	163.416983	87201.37	46392.64	103097.39	69982.83	158641.23	135739.53	132042.47	91173.17	30233.95	19199.79	1.302479e+07	7.015004e+06	1.573184e+07	1.077617e+07	2.470464e+07	2.132618e+07	2.100419e+07	1.460464e+07	4.896294e+06	3.137572e+06	2025-06-01
2	2025-06-01 00:01:24	149.346883	151.163073	152.550295	153.948960	155.680957	157.069958	159.009792	160.158561	161.929228	163.406900	90172.12	48761.71	101064.41	76837.58	159411.74	129815.46	138608.64	95230.38	30510.96	19339.21	1.346693e+07	7.370970e+06	1.541741e+07	1.182907e+07	2.481737e+07	2.039011e+07	2.204013e+07	1.525196e+07	4.940616e+06	3.160160e+06	2025-06-01
3	2025-06-01 00:01:55	149.295042	150.905086	152.424913	154.030585	155.593302	156.944807	158.944380	160.065701	161.778058	163.337892	89323.49	39773.81	109516.79	102535.13	161776.85	134194.11	134831.70	98632.40	33391.53	20464.60	1.333555e+07	6.002070e+06	1.669309e+07	1.579355e+07	2.517139e+07	2.106107e+07	2.143074e+07	1.578766e+07	5.402017e+06	3.342645e+06	2025-06-01
4	2025-06-01 00:02:14	149.325217	150.953011	152.432532	154.050369	155.620230	156.969589	158.959332	160.101163	161.876147	163.351747	85072.79	36722.47	108171.85	101794.40	166286.48	132158.35	133991.31	100434.25	30265.46	20148.63	1.270351e+07	5.543367e+06	1.648891e+07	1.568146e+07	2.587754e+07	2.074484e+07	2.129917e+07	1.607964e+07	4.899256e+06	3.291314e+06	2025-06-01
请你帮我编写代码，对每个timestamp，计算过去100个不同的timestamp中的全量订单簿各档位变化量（当前api - 过去api, 当前bpi-过去bpi）。
注意：全局订单簿变化量指当前timestamp ap1-ap5 与 100个timestamp之前的ap1-ap5 按照买方从大到小，卖方从小到大排序后，当前api - 过去api i = 1..x x为当前和过去ap1-ap5 unique 的个数）
挂单的筹码分布skew 时间做一个decay，偏度的变化量，（全量订单簿变化量的偏度， 最后一个时刻的偏度）

挂单的筹码：价格上涨：新增档位委买量+原档位净委买增加量+主买成交额； 被动卖出成交额+原档位净委卖减少量

新增档位委买量 / 被动卖出成交额；
（新增档位委买量 + 净主买成交额） /  净被动卖出成交额

（新增档位委买量 + 原档位净委买增加量） / （被动卖出成交额 + 原档位净委卖减少量）

（新增档位委买量 + 原档位净委买增加量 + 净主买成交额） / （净被动卖出成交额 + 原档位净委卖减少量）

全量订单簿变化量

